# Template Eksperimen MSML

Notebook ini mengikuti alur eksperimen manual untuk dataset **Customer Churn**: data loading, EDA, preprocessing, dan penyimpanan dataset siap latih.

## 1. Import Library
Library yang digunakan meliputi pandas, numpy, matplotlib, seaborn, dan scikit-learn untuk preprocessing.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

## 2. Data Loading
Dataset raw dibaca dari folder `customer_churn_raw`.

In [2]:
DATA_PATH = '../customer_churn_raw/customer_churn_raw.csv'
df = pd.read_csv(DATA_PATH)
print('Ukuran data:', df.shape)
df.head()

Ukuran data: (7043, 21)


  customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

  MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection  \
0      No phone service             DSL             No          Yes               No   
1                    No             DSL            Yes           No              Yes   
2                    No             DSL            Yes          Yes               No   
3      No phone service             DSL            Yes           No              Yes   
4                    No     Fiber optic             No           No               No   


## 3. Exploratory Data Analysis (EDA)
Tahap ini melihat struktur data, missing value, distribusi target, dan karakteristik fitur utama.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.isna().sum().sort_values(ascending=False).head(10)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
dtype: int64

In [5]:
df['Churn'].value_counts(normalize=True).mul(100).round(2)

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

In [6]:
df[['tenure','MonthlyCharges']].describe()

            tenure  MonthlyCharges
count  7043.000000     7043.000000
mean     32.371149       64.761692
std      24.559481       30.090047
min       0.000000       18.250000
25%       9.000000       35.500000
50%      29.000000       70.350000
75%      55.000000       89.850000
max      72.000000      118.750000

## 4. Data Preprocessing
Tahapan preprocessing meliputi penghapusan ID pelanggan, konversi `TotalCharges`, imputasi missing value, encoding target, one-hot encoding, dan scaling fitur numerik.

In [7]:
df_clean = df.copy()

if 'customerID' in df_clean.columns:
    df_clean = df_clean.drop(columns=['customerID'])

df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median())

df_clean['Churn'] = df_clean['Churn'].map({'No': 0, 'Yes': 1}).astype(int)

cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_encoded[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print('Ukuran data setelah preprocessing:', df_encoded.shape)
df_encoded.head()

Ukuran data setelah preprocessing: (7043, 31)


   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  Churn  gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  MultipleLines_No phone service  MultipleLines_Yes  InternetService_Fiber optic  InternetService_No  OnlineSecurity_No internet service  OnlineSecurity_Yes  OnlineBackup_No internet service  OnlineBackup_Yes  DeviceProtection_No internet service  DeviceProtection_Yes  TechSupport_No internet service  TechSupport_Yes  StreamingTV_No internet service  StreamingTV_Yes  StreamingMovies_No internet service  StreamingMovies_Yes  Contract_One year  Contract_Two year  PaperlessBilling_Yes  PaymentMethod_Credit card (automatic)  PaymentMethod_Electronic check  PaymentMethod_Mailed check
0              0       1           29.85         29.85      0        False         True           False             False                            True              False                        False               False                               False               False                

## 5. Menyimpan Dataset Preprocessing
Dataset hasil preprocessing disimpan agar dapat digunakan pada tahap modelling dan workflow CI.

In [8]:
import os

OUTPUT_PATH = 'customer_churn_preprocessing/churn_preprocessed.csv'
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_encoded.to_csv(OUTPUT_PATH, index=False)

print(f'Dataset hasil preprocessing tersimpan di: {OUTPUT_PATH}')
print('Final shape:', df_encoded.shape)

Dataset hasil preprocessing tersimpan di: customer_churn_preprocessing/churn_preprocessed.csv
Final shape: (7043, 31)


## 6. Kesimpulan Eksperimen
Dataset telah melalui tahapan loading, EDA, dan preprocessing. File hasil preprocessing siap digunakan untuk pelatihan model pada tahap berikutnya.